# Стеки, очереди, деки, связные списки: ООП-подход

**Цель урока:** научиться моделировать основные структуры данных через классы Python и понимать, где они применяются.

**После урока вы сможете:**
- реализовать стек, очередь, дек, односвязный и двусвязный списки;
- объяснять сложность операций O(1)/O(n);
- применять структуры в практических задачах: история браузера, очередь печати, плейлист, LRU-кэш;
- решать самостоятельные задачи.

**Как работать с ноутбуком:** запускайте ячейки по порядку (`Run All` или `Shift+Enter`). Код написан так, чтобы его можно было читать и модифицировать.

## 1. ООП и абстрактные типы данных

**Абстрактный тип данных (ADT)** описывает *что* структура умеет, а не *как* хранит данные. Например, стек должен поддерживать `push`, `pop`, `peek`.

**ООП** помогает скрыть внутреннее устройство. В классе Python:
- `__init__` создает объект;
- обычные методы реализуют операции;
- «дундер»-методы (`__len__`, `__repr__`, `__iter__`) делают объект удобным;
- защищенные атрибуты с `_` сигнализируют, что напрямую к ним лучше не обращаться.

Ниже для каждой структуры сначала теория, затем класс, затем примеры и проверки через `assert`.

## 2. Стек (Stack)

**Стек** работает по принципу **LIFO** (Last In, First Out): последним пришел — первым ушел.

Основные операции:
- `push(x)` — положить элемент наверх;
- `pop()` — снять верхний элемент;
- `peek()` — посмотреть верхний элемент, не удаляя;
- `is_empty()` — проверить, пуст ли стек.

В типовой реализации на динамическом массиве (`list`) все операции амортизированно выполняются за **O(1)**.

**Где применяется:** вызов функций, отмена действий (undo), проверка сбалансированности скобок, обход графов в глубину (DFS).

In [3]:
class StackError(Exception):
    pass

class Stack:
    def __init__(self):
        self._data = []

    def clear(self):
        self._data = []

    def push(self, item):
        self._data.append(item)

    def pop(self):
        if self.is_empty():
            raise StackError('pop from empty stack')
        return self._data.pop()

    def peek(self):
        if self.is_empty():
            raise StackError('peek from empty stack')
        return self._data[-1]

    def is_empty(self):
        return len(self._data) == 0

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f'Stack({self._data})'

# Мини-тесты
s = Stack()
assert s.is_empty()
s.push(10)
s.push(20)
s.push(30)
assert len(s) == 3
assert s.peek() == 30
assert s.pop() == 30
assert s.pop() == 20
assert len(s) == 1

### Практический пример: скобки и редактор с undo/redo

1. Проверка сбалансированности скобок — классическая задача для стека.
2. Редактор текста хранит последние изменения в стеке отмены. При новом вводе история повторов (`redo`) очищается.

In [4]:
def is_balanced(expression):
    stack = Stack()
    pairs = {')': '(', ']': '[', '}': '{'}
    opening = set(pairs.values())
    for ch in expression:
        if ch in opening:
            stack.push(ch)
        elif ch in pairs:
            if stack.is_empty() or stack.pop() != pairs[ch]:
                return False
    return stack.is_empty()

assert is_balanced('(a + [b * {c}])')
assert not is_balanced('(a + [b * {c})]')

class TextEditor:
    def __init__(self):
        self.text = ''
        self.undo_stack = Stack()
        self.redo_stack = Stack()

    def type(self, chars):
        if not chars:
            return
        self.text += chars
        # self.undo_stack.append(chars)
        self.undo_stack.push(chars)
        self.redo_stack.clear()

    def undo(self):
        if self.undo_stack.is_empty():
            return
        chars = self.undo_stack.pop()
        self.text = self.text[:-len(chars)]
        self.redo_stack.push(chars)

    def redo(self):
        if self.redo_stack.is_empty():
            return
        chars = self.redo_stack.pop()
        self.text += chars
        self.undo_stack.push(chars)

    def __repr__(self):
        return f'TextEditor(text={self.text!r})'

editor = TextEditor()
editor.type('Hello')
editor.type(' World')
assert editor.text == 'Hello World'
editor.undo()
assert editor.text == 'Hello'
editor.redo()
assert editor.text == 'Hello World'

## 3. Очередь (Queue)

**Очередь** работает по принципу **FIFO** (First In, First Out): первым пришел — первым ушел.

Основные операции:
- `enqueue(x)` — добавить в конец;
- `dequeue()` — взять из начала;
- `front()` — посмотреть первый элемент;
- `is_empty()` — проверка пустоты.

Для эффективной очереди в Python удобно использовать `collections.deque`, потому что `popleft()` у списка был бы **O(n)**.

**Где применяется:** очередь печати, планировщики задач, BFS, обработка запросов, моделирование очередей.

In [5]:
from collections import deque

class Queue:
    def __init__(self):
        self._data = deque()

    def enqueue(self, item):
        self._data.append(item)

    def dequeue(self):
        if self.is_empty():
            raise IndexError('dequeue from empty queue')
        return self._data.popleft()

    def front(self):
        if self.is_empty():
            raise IndexError('front from empty queue')
        return self._data[0]

    def is_empty(self):
        return len(self._data) == 0

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f'Queue({list(self._data)})'

q = Queue()
q.enqueue('job-1')
q.enqueue('job-2')
assert q.front() == 'job-1'
assert q.dequeue() == 'job-1'
assert q.dequeue() == 'job-2'
assert q.is_empty()

### Практический пример: принтер и моделирование очереди

Принтер обрабатывает задания строго по порядку поступления. Каждое задание можно хранить как кортеж `(имя, количество страниц)`.

In [6]:
class Printer:
    def __init__(self):
        self.queue = Queue()
        self.completed = []

    def add_job(self, name, pages):
        self.queue.enqueue((name, pages))

    def print_next(self):
        if self.queue.is_empty():
            return None
        name, pages = self.queue.dequeue()
        record = f'{name}: {pages} стр.'
        self.completed.append(record)
        return record

printer = Printer()
printer.add_job('Отчет', 10)
printer.add_job('Договор', 3)
assert printer.print_next() == 'Отчет: 10 стр.'
assert printer.print_next() == 'Договор: 3 стр.'
assert printer.print_next() is None
assert printer.completed == ['Отчет: 10 стр.', 'Договор: 3 стр.']

## 4. Дек (Deque)

**Дек** (double-ended queue) позволяет добавлять и удалять элементы с обоих концов.

Основные операции:
- `add_left(x)`, `add_right(x)`;
- `pop_left()`, `pop_right()`;
- `peek_left()`, `peek_right()`.

Все перечисленные операции в хорошей реализации выполняются за **O(1)**.

**Где применяется:** скользящее окно, палиндромы, планирование с доступом к обоим концам, undo/redo с историей, work-stealing в многопоточных системах.

In [7]:
class Deque:
    def __init__(self):
        self._data = deque()

    def add_left(self, item):
        self._data.appendleft(item)

    def add_right(self, item):
        self._data.append(item)

    def pop_left(self):
        if self.is_empty():
            raise IndexError('pop from empty deque')
        return self._data.popleft()

    def pop_right(self):
        if self.is_empty():
            raise IndexError('pop from empty deque')
        return self._data.pop()

    def peek_left(self):
        if self.is_empty():
            raise IndexError('peek from empty deque')
        return self._data[0]

    def peek_right(self):
        if self.is_empty():
            raise IndexError('peek from empty deque')
        return self._data[-1]

    def is_empty(self):
        return len(self._data) == 0

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f'Deque({list(self._data)})'

d = Deque()
d.add_right(2)
d.add_right(3)
d.add_left(1)
assert len(d) == 3
assert d.pop_left() == 1
assert d.pop_right() == 3
assert d.peek_left() == 2

### Практический пример: палиндром и скользящее максимум-окно

- Палиндром можно проверять, снимая символы с левого и правого концов дека.
- Задача «максимум в скользящем окне» решается с помощью дека индексов, где значения монотонно убывают.

In [8]:
def is_palindrome(text):
    d = Deque()
    letters = [ch.lower() for ch in text if ch.isalnum()]
    for ch in letters:
        d.add_right(ch)
    while len(d) > 1:
        if d.pop_left() != d.pop_right():
            return False
    return True

assert is_palindrome('А роза упала на лапу Азора')
assert not is_palindrome('Информатика')

def max_sliding_window(nums, k):
    if k <= 0:
        return []
    window = deque()
    result = []
    for i, num in enumerate(nums):
        while window and nums[window[-1]] <= num:
            window.pop()
        window.append(i)
        if window[0] <= i - k:
            window.popleft()
        if i >= k - 1:
            result.append(nums[window[0]])
    return result

assert max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]

## 5. Односвязный список (Singly Linked List)

**Односвязный список** состоит из узлов. Каждый узел хранит значение и ссылку на следующий узел. Последний узел ссылается на `None`.

Достоинства:
- вставка/удаление в начале за **O(1)**;
- нет необходимости заранее знать размер;
- удобно строить очереди и стеки поверх узлов.

Недостатки:
- доступ по индексу за **O(n)**;
- нельзя быстро идти назад.

**Где применяется:** плейлист «дальше», цепочки задач, списки смежности в графах, реализации очередей.

In [13]:
class SNode:
    def __init__(self, value, next=None):
        self.value = value
        self.next = next

    def __repr__(self):
        return f'SNode({self.value!r})'

class SinglyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self._size = 0

    def get_head(self):
        return self.head

    def get_tail(self):
        return self.tail

    def append(self, value):
        node = SNode(value)
        if self.head is None:
            self.head = node
            self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self._size += 1

    def prepend(self, value):
        node = SNode(value, self.head)
        self.head = node
        if self.tail is None:
            self.tail = node
        self._size += 1

    def insert_at(self, index, value):
        if index < 0 or index > self._size:
            raise IndexError('index out of range')
        if index == 0:
            self.prepend(value)
            return
        if index == self._size:
            self.append(value)
            return
        current = self.head
        for _ in range(index - 1):
            current = current.next
        current.next = SNode(value, current.next)
        self._size += 1

    def delete_at(self, index):
        if index < 0 or index >= self._size:
            raise IndexError('index out of range')
        if index == 0:
            value = self.head.value
            self.head = self.head.next
            if self.head is None:
                self.tail = None
            self._size -= 1
            return value
        prev = self.head
        for _ in range(index - 1):
            prev = prev.next
        target = prev.next
        prev.next = target.next
        if target is self.tail:
            self.tail = prev
        self._size -= 1
        return target.value

    def remove(self, value):
        prev = None
        current = self.head
        while current is not None:
            if current.value == value:
                if prev is None:
                    self.head = current.next
                else:
                    prev.next = current.next
                if current is self.tail:
                    self.tail = prev
                self._size -= 1
                return True
            prev = current
            current = current.next
        return False

    def search(self, value):
        index = 0
        current = self.head
        while current is not None:
            if current.value == value:
                return index
            current = current.next
            index += 1
        return -1

    def __iter__(self):
        current = self.head
        while current is not None:
            yield current.value
            current = current.next

    def __len__(self):
        return self._size

    def __repr__(self):
        return 'SinglyLinkedList([' + ', '.join(repr(x) for x in self) + '])'

ll = SinglyLinkedList()
ll.append(1)
ll.append(2)
ll.prepend(0)
ll.insert_at(2, 99)
assert list(ll) == [0, 1, 99, 2]
assert ll.delete_at(2) == 99
assert ll.remove(1) is True
assert list(ll) == [0, 2]
assert ll.search(2) == 1
assert len(ll) == 2